# 3D Gaussian Splatting scan - free GPU (T4) - proof of concept

**What this is:** the *scan-pipeline proof of concept* for the farm digital
twin: photos -> 3DGS training -> `.ply` -> USD (`scan_io`) -> browser viewer.

**What this is NOT:** a farm scan. The capture is the INRIA public reference
scene (`truck` - outdoor, vegetation around it). Real field scans are the
post-hackathon Phase 3 item. Labeled honestly; see `docs/04` D2.

**Runs on:** free Kaggle (T4 x2, internet ON in notebook options) or free
Colab T4. Time budget: compile ~8-12 min, data ~3 min, training ~12-20 min
(7k iters), orbit render ~2-4 min.
**No COLMAP needed** - the dataset ships its SfM poses (`sparse/0`).

**Artifacts written to the working dir:** `scan.ply` (full), `scan_viewer.ply`
(capped for the web viewer), `farm_scan.usda` (OpenUSD), `orbit.mp4`,
`metrics.json`. Big scratch (repo clone, dataset, training run) lives in
`/tmp/kgs` on Kaggle so the kernel output stays small.


## 0. Environment probe + GPU check

In [ ]:
import os
ON_KAGGLE = os.path.isdir("/kaggle/working")
WORK = os.getcwd()
# Big scratch (clone, dataset, training output) outside the persisted dir on
# Kaggle: the kernel output then holds only the final artifacts.
BASE = "/tmp/kgs" if ON_KAGGLE else WORK
os.makedirs(BASE, exist_ok=True)
os.chdir(BASE)
print("env:", "Kaggle" if ON_KAGGLE else "Colab/local", "| WORK:", WORK, "| BASE:", BASE)
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable the GPU accelerator (Kaggle: options > accelerator > GPU T4 x2; Colab: Runtime > Change runtime type > T4), then rerun from the top"


## 1. 3DGS trainer (INRIA reference implementation, current HEAD)
`--recursive` is mandatory: the three CUDA extensions are git submodules
(`diff-gaussian-rasterization` @ `dr_aa`, `simple-knn`, `fused-ssim`).
Compile: ~8-12 min, once per session.
`usd-core==26.8` + `imageio-ffmpeg` + `plyfile` cover the conversion/render
cells later in this notebook.


In [ ]:
if not os.path.isdir("gauss"):
    !git clone --depth 1 --recursive https://github.com/graphdeco-inria/gaussian-splatting.git gauss
os.chdir("gauss")
!pip install -q plyfile tqdm imageio imageio-ffmpeg "usd-core==26.8"
!pip install -q submodules/diff-gaussian-rasterization submodules/simple-knn submodules/fused-ssim
import diff_gaussian_rasterization, simple_knn, fused_ssim
print("trainer ready:", diff_gaussian_rasterization.__file__)


## 2. Data: public reference capture (SfM poses included)
~650 MB download. The archive contains `tandt/truck`, `tandt/train`, `db/*`;
each scene ships `sparse/0` COLMAP models, so there is **no COLMAP step**.


In [ ]:
os.chdir("..")
import subprocess
if not os.path.isdir("tandt/truck"):
    url = "https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip"
    subprocess.run(["wget", "-q", url], check=True)          # ~650 MB
    subprocess.run(["unzip", "-q", "tandt_db.zip"], check=True)
assert os.path.exists("tandt/truck/sparse/0/images.bin"), "COLMAP sparse model expected"
n = len(os.listdir("tandt/truck/images"))
print(f"truck: {n} images + COLMAP poses ready (no SfM needed)")


## 3. Train (7k iterations)
~12-20 min on T4. NO `--quiet`: at repo HEAD `safe_state(silent=True)
discards every print(), which would hide the PSNR evaluation lines we parse
below. Flags verified against HEAD (`--test_iterations`, `--disable_viewer`).
The cell fails loudly (with the log tail) if the trainer exits nonzero.


In [ ]:
os.chdir("gauss")
import subprocess
cmd = ("python train.py -s ../tandt/truck -m ../out/truck --iterations 7000 "
       "--test_iterations 3500 7000 --save_iterations 7000 --disable_viewer 2>&1 | tee ../train.log")
r = subprocess.run(["bash", "-lc", cmd])
print("train exit code:", r.returncode)
assert r.returncode == 0, "training failed - tail of train.log:\n" + open("../train.log").read()[-2500:]
print("training done")


## 4. Metrics
Log format at HEAD: `[ITER n] Evaluating <split>: L1 <x> PSNR <y> [date]`.


In [ ]:
import re, json
log = open("../train.log").read()
pairs = re.findall(r"\[ITER (\d+)\] Evaluating (\w+): L1 [\d.eE+-]+ PSNR ([\d.]+)", log)
metrics = {"psnr": {f"{it}_{split}": float(v) for it, split, v in pairs}}
if not metrics["psnr"]:
    raise RuntimeError("no PSNR lines parsed - train.log tail:\n" + log[-2500:])
print(json.dumps(metrics, indent=1))
json.dump(metrics, open("../metrics.json", "w"))


## 5. Flythrough video (novel views, real poses)
Renders the training cameras in angular order around the scene center -
a guaranteed-good orbit without custom projection code, using the repo's own
rasterizer at HEAD (`antialiasing` pipeline flag, trained-exposure off).
Fallback: skip this cell; the PLY + viewer still demo fine.


In [ ]:
import math, torch, imageio.v2 as imageio
from argparse import ArgumentParser
from scene import Scene, GaussianModel
from gaussian_renderer import render
from arguments import ModelParams, PipelineParams

parser = ArgumentParser()
mp = ModelParams(parser); pp = PipelineParams(parser)
args = parser.parse_args(["-s", "../tandt/truck", "-m", "../out/truck"])
dataset, pipe = mp.extract(args), pp.extract(args)
gaussians = GaussianModel(dataset.sh_degree)
scene = Scene(dataset, gaussians, load_iteration=-1, shuffle=False)
bg = torch.tensor([0.0, 0.0, 0.0], dtype=torch.float32, device="cuda")
cams = scene.getTrainCameras()
center = torch.stack([c.camera_center for c in cams]).mean(0)

def azim(c):
    d = c.camera_center - center
    return torch.atan2(d[1], d[0]).item()

order = sorted(range(len(cams)), key=lambda i: azim(cams[i]))
MAX_FRAMES = 160
if len(order) > MAX_FRAMES:
    step = len(order) / MAX_FRAMES
    order = [order[int(i * step)] for i in range(MAX_FRAMES)]
frames = []
with torch.no_grad():
    for i in order:
        out = render(cams[i], gaussians, pipe, bg, use_trained_exp=dataset.train_test_exp)
        img = (torch.clamp(out["render"], 0, 1).permute(1, 2, 0) * 255).byte().cpu().numpy()
        frames.append(img)
# libx264/yuv420p needs even dimensions: truck frames are 1243x822 (odd width).
h, w = frames[0].shape[:2]
h2, w2 = h - (h % 2), w - (w % 2)
frames = [f[:h2, :w2] for f in frames]
imageio.mimwrite("../orbit.mp4", frames, fps=24, macro_block_size=None, quality=7)
print(f"orbit.mp4: {len(frames)} frames @ {w2}x{h2},",
      round(os.path.getsize('../orbit.mp4') / 1e6, 1), "MB")
print("gaussians in scan:", gaussians.get_xyz.shape[0])


## 6. Export artifacts (PLY -> viewer PLY + OpenUSD)
`scan.ply` - full trained model. `scan_viewer.ply` - stride-capped to 150k
splats so the browser viewer stays responsive and the file fits GitHub
(< 40 MB). `farm_scan.usda` - the same scan converted to OpenUSD
`UsdGeomPoints` by the twin repo's `scan_io` (SH-C0 colors, clamped widths,
Y-up -> Z-up), composing headless into the farm layer stack. On Kaggle the
five artifacts are copied into `/kaggle/working` (the kernel output).


In [ ]:
import shutil, subprocess, math
from plyfile import PlyData, PlyElement

ply = "../out/truck/point_cloud/iteration_7000/point_cloud.ply"
shutil.copy(ply, "../scan.ply")

# Twin repo source: attached as a Kaggle dataset, or cloned on Colab.
TWIN_SRC = "/kaggle/input/3dcrop-twin-src" if ON_KAGGLE else "../twin"
if not ON_KAGGLE and not os.path.isdir(TWIN_SRC):
    subprocess.run(["git", "clone", "--depth", "1", "-b", "pivot",
                    "https://github.com/Only1allan/3d_crop_Intelligence-v0.git", TWIN_SRC], check=True)
sys_path = f"{TWIN_SRC}/exts/omniverse_crop_intelligence"
import sys; sys.path.insert(0, sys_path)
from omniverse_crop_intelligence.scan_io import convert
summary = convert("../scan.ply", "../farm_scan.usda", max_points=150_000)

src = PlyData.read("../scan.ply")
n = src["vertex"].count
stride = max(1, math.ceil(n / 150_000))
PlyData([PlyElement.describe(src["vertex"][::stride], "vertex")], text=False).write("../scan_viewer.ply")

ARTIFACTS = ("scan.ply", "scan_viewer.ply", "farm_scan.usda", "orbit.mp4", "metrics.json")
if ON_KAGGLE:
    for f in ARTIFACTS:
        shutil.copy(f"../{f}", os.path.join(WORK, f))
for f in ARTIFACTS:
    p = os.path.join(WORK, f) if ON_KAGGLE else f"../{f}"
    print(f"{f:>16}: {os.path.getsize(p)/1e6:8.1f} MB")
print("usda:", summary["points_out"], "of", summary["points_in"], "points | widths:", summary["widths"])


## Troubleshooting
| symptom | fix |
|---|---|
| CUDA OOM during training | rerun training with `--iterations 5000`; T4 fits this scene comfortably at 7k |
| submodule compile error | delete the `gauss` folder (in `BASE`), rerun cell 1 (the clone must be `--recursive`) |
| `unrecognized arguments: --testing_iterations` | you are on a stale fork - use graphdeco-inria HEAD, flags were renamed |
| no PSNR lines parsed | do not add `--quiet`: HEAD's `safe_state(silent)` drops all prints, incl. the evaluation lines |
| no GPU available | Kaggle: check weekly GPU quota (Settings); Colab: retry later |
| download stalls | the dataset is re-fetchable; rerun the data cell |
| Kaggle private dataset error | make sure `3dcrop-twin-src` belongs to the same account as the kernel |
